# 01 - Exploratory data analysis

**Presentation layer only.** Every number and figure below is produced by `src/eda.py`
and `src/figures.py`. This notebook defines no analysis logic of its own, and re-running
it cannot change a result. The command-line equivalent is:

```bash
python run_all.py --stage eda
```

## The question this notebook answers

Does SPY's daily return series actually exhibit the conditional heteroskedasticity that
a GARCH model exists to capture? The whole project rests on the answer being yes, so it
is established by test rather than by pointing at a plot that looks clustered.

## Which window

Primary results use the **training window only** (2014-01-02 to 2016-12-30). Justifying
the model class with a statistic computed over the out-of-sample period would let the
evaluation window argue for the model later evaluated on it. Full-sample values appear
at the end as descriptive context and justify nothing.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Image, display

from src import eda, figures
from src.data import TRAIN_END, TRAIN_START

PROJECT_ROOT = Path.cwd().parent
frame = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "analysis_frame.csv",
    index_col=0,
    parse_dates=True,
)
print(f"analysis frame: {len(frame):,} rows, {frame.index.min().date()} to {frame.index.max().date()}")
frame.head()

## 1. The data

Three quantities that must never be conflated: the observed **return** (the calibration
target), the **Parkinson** high/low proxy (the point-forecast target only), and the
**lagged VIX close** (the regime label).

In [ ]:
frame["regime"].value_counts().reindex(["calm", "normal", "stressed"]).to_frame("days")

The stressed regime is a minority of the sample. Every regime-conditional table in
this project reports `n` alongside its statistic for that reason -- and Stage 5 attaches
bootstrap confidence intervals, because a coverage estimate on a few hundred days is not
a precise number.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / "figures" / "01_returns_clustering.png")))

Large moves arrive next to other large moves and quiet days next to quiet days. That
is volatility clustering, and it is the single stylised fact GARCH parameterises. The
shaded band is the training window; everything to its right is out of sample.

## 2. The motivating tests

Computed on the training window.

In [ ]:
train_returns = frame.loc[TRAIN_START:TRAIN_END, "log_return"]
train_report = eda.run_eda(train_returns, window_label="training window (PRIMARY)")
print(train_report.format_full())

Read the three blocks together, because the contrast is the finding:

- **ARCH-LM rejects at every lag.** Volatility is predictable from its own past.
- **Ljung-Box on squared returns rejects overwhelmingly.** The same fact from another
  angle.
- **Ljung-Box on raw returns does not reject at any lag.** The *mean* is close to
  unpredictable.

Structure in the second moment, almost none in the first: that is the regime in which a
conditional-variance model earns its place, and it is why this project forecasts
variance rather than direction.

In [ ]:
print("VERDICT:", train_report.headline_verdict())

In [ ]:
display(Image(filename=str(PROJECT_ROOT / "figures" / "02_squared_return_acf.png")))

The picture behind the two tests: autocorrelation in squared returns is well outside
the no-autocorrelation band for roughly the first eleven lags and decays slowly. An
i.i.d. series would leave almost every spike inside the grey band.

## 3. Fat tails

The reason the locked design specifies a Student-t innovation rather than a normal.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / "figures" / "03_return_distribution.png")))
print(f"excess kurtosis (training window): {train_report.summary.excess_kurtosis:.3f}")

Excess kurtosis is materially positive and both tails sit above the matched normal.
A Gaussian innovation would systematically understate the probability of large moves --
which is exactly the 99% VaR failure the Stage 6 ablation is designed to expose.

## 4. Regimes

Thresholds are fixed ex ante at VIX 15 and 25 and applied to the **lagged** close, so no
regime label can depend on the day it labels.

In [ ]:
display(Image(filename=str(PROJECT_ROOT / "figures" / "04_vix_regimes.png")))

## 5. Full sample -- descriptive context only

Reported so a reader can see it agrees with the training window. It justifies nothing:
every design decision was locked before these numbers existed (`research_log.md` 1.1).

In [ ]:
full_report = eda.run_eda(frame["log_return"], window_label="full sample (DESCRIPTIVE)")
print(full_report.format_full())

**One difference is worth noting rather than passing over.** On the full sample,
Ljung-Box on *raw* returns rejects, where on the training window it does not. Short-horizon
return autocorrelation rises in crises, and the out-of-sample period contains several.

This does not bias the project's comparison: a constant mean is applied identically to
all four forecasters, so it cannot favour one over another. It does mean the constant-mean
assumption is a real simplification over the evaluation period, and it belongs in the
report's limitations section rather than being quietly omitted.